# Chapter 1 — Spherical mass and the effective potential

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vasilybelokurov/galactic-dynamics-tutorial/blob/main/notebooks/01_spherical_mass.ipynb)

**Goal.** Before integrating a single orbit, learn to *predict* its key properties:
the gravitational field of a spherical body, its circular speed, and the radii between
which a star oscillates.

**You will learn**
1. the shell theorem — why only the mass *inside* radius $r$ pulls on a star at $r$;
2. how density $\rho(r)$ determines enclosed mass $M(r)$, potential $\Phi(r)$ and circular speed $v_c(r)$;
3. the effective potential $\Phi_{\rm eff}(r)$ and how it fixes pericentre and apocentre;
4. the virial theorem as a global check on a self-gravitating system.

**Units.** Throughout this chapter $G = 1$, and masses and lengths are in units of the
model's total mass $M$ and scale radius $b$. (Chapter 3 shows how to convert to kpc, km/s, M$_\odot$.)

**Prerequisites.** Newton's law of gravity, vectors, basic integrals; Python with `numpy` and `matplotlib`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import quad
from scipy.optimize import brentq, minimize_scalar

G = 1.0

## 1.1 The shell theorem, checked numerically

Newton showed that a thin spherical shell of mass $m$ and radius $R$
- exerts **no net force** on a point inside it ($r < R$), and
- attracts a point outside it ($r > R$) **as if all its mass were at the centre**: $g = G m / r^2$.

Rather than take this on trust, we build a shell out of $N$ point masses and add up
their pulls directly. To spread points evenly over a sphere we use the
*Fibonacci lattice*: point $i$ sits at height $z_i = 1 - (2i+1)/N$ and azimuth
$\phi_i = i \times 137.5^\circ$ (the "golden angle").

In [ ]:
def fibonacci_sphere(n, radius=1.0):
    """n points spread almost uniformly over a sphere; returns array of shape (n, 3)."""
    i = np.arange(n)
    z = 1 - (2 * i + 1) / n
    phi = i * np.pi * (3 - np.sqrt(5))          # golden angle in radians
    s = np.sqrt(1 - z**2)
    return radius * np.column_stack([s * np.cos(phi), s * np.sin(phi), z])


def acceleration(point, sources, masses):
    """Newtonian acceleration at `point` due to point masses at `sources`."""
    d = sources - point                          # vectors from the point to each mass
    r3 = np.sum(d**2, axis=1)**1.5
    return G * np.sum(masses[:, None] * d / r3[:, None], axis=0)


N, m_shell, R = 20000, 1.0, 1.0
shell = fibonacci_sphere(N, R)
masses = np.full(N, m_shell / N)

# radial pull g_r at test points along a direction that avoids the lattice points
direction = np.array([0.3, 0.5, 0.81]) / np.linalg.norm([0.3, 0.5, 0.81])
r_test = np.concatenate([np.linspace(0.05, 0.9, 12), np.linspace(1.1, 4, 12)])
g_r = np.array([acceleration(r * direction, shell, masses) @ direction for r in r_test])

plt.figure(figsize=(6, 4))
plt.plot(r_test, -g_r, 'o', label=f'sum over {N} point masses')
rr = np.linspace(1.0, 4, 100)
plt.plot(rr, G * m_shell / rr**2, 'k-', label=r'$Gm/r^2$ (shell theorem)')
plt.plot([0, 1], [0, 0], 'k-')
plt.axvline(R, color='gray', ls=':')
plt.xlabel('distance from centre $r$'); plt.ylabel('inward pull $-g_r$')
plt.legend(); plt.title('Field of a thin shell of radius 1'); plt.show()

inside, outside = r_test < R, r_test > R
print('max |g| inside             :', np.max(np.abs(g_r[inside])))
print('max relative error outside :', np.max(np.abs(-g_r[outside] * r_test[outside]**2 / (G * m_shell) - 1)))
assert np.max(np.abs(g_r[inside])) < 1e-5
assert np.max(np.abs(-g_r[outside] * r_test[outside]**2 / (G * m_shell) - 1)) < 1e-5

Inside the shell the pulls cancel to about one part in a million; outside they add up to $Gm/r^2$
to the same accuracy. The residuals come from representing a smooth shell by a finite number of points.

**Consequence.** A spherical galaxy is a set of nested shells. A star at radius $r$ feels only
the shells inside it:

$$ g(r) = -\frac{G M(r)}{r^2}, \qquad M(r) = 4\pi \int_0^r \rho(r')\, r'^2\, dr'. $$

The potential $\Phi$ (with $\mathbf g = -\nabla\Phi$ and $\Phi \to 0$ at infinity) collects
both the inner mass and the outer shells, each of which contributes a constant $-Gm/R$ inside it:

$$ \Phi(r) = -\frac{G M(r)}{r} - 4\pi G \int_r^\infty \rho(r')\, r'\, dr'. $$

## 1.2 From density to potential: the Plummer sphere

The **Plummer sphere** (Plummer 1911) is the simplest smooth model of a star cluster or
galaxy core:

$$ \rho(r) = \frac{3M}{4\pi b^3}\left(1+\frac{r^2}{b^2}\right)^{-5/2}, \qquad
   M(r) = \frac{M r^3}{(r^2+b^2)^{3/2}}, \qquad
   \Phi(r) = -\frac{GM}{\sqrt{r^2+b^2}}. $$

We compute $M(r)$ and $\Phi(r)$ *numerically* from $\rho$ with the two integrals above and
compare them with these formulae. This is the pattern we will use throughout:
**a numerical method is trusted only after it reproduces a known answer.**

In [ ]:
M, b = 1.0, 1.0

def rho_plummer(r):
    return 3 * M / (4 * np.pi * b**3) * (1 + r**2 / b**2)**-2.5

def mass_numeric(r, rho):
    return quad(lambda x: 4 * np.pi * rho(x) * x**2, 0, r)[0]

def phi_numeric(r, rho):
    outer = quad(lambda x: rho(x) * x, r, np.inf)[0]
    return -G * mass_numeric(r, rho) / r - 4 * np.pi * G * outer

r = np.logspace(-2, 2, 60)
M_num = np.array([mass_numeric(x, rho_plummer) for x in r])
Phi_num = np.array([phi_numeric(x, rho_plummer) for x in r])
M_exact = M * r**3 / (r**2 + b**2)**1.5
Phi_exact = -G * M / np.sqrt(r**2 + b**2)

fig, ax = plt.subplots(1, 3, figsize=(13, 3.8))
ax[0].loglog(r, rho_plummer(r), 'k-'); ax[0].set_ylabel(r'$\rho(r)$')
ax[1].loglog(r, M_exact, 'k-', lw=3, alpha=0.3, label='formula')
ax[1].loglog(r, M_num, 'r--', label='numerical'); ax[1].set_ylabel('$M(r)$'); ax[1].legend()
ax[2].semilogx(r, Phi_exact, 'k-', lw=3, alpha=0.3, label='formula')
ax[2].semilogx(r, Phi_num, 'r--', label='numerical'); ax[2].set_ylabel(r'$\Phi(r)$'); ax[2].legend()
for a in ax: a.set_xlabel('$r/b$')
plt.tight_layout(); plt.show()

err_M = np.max(np.abs(M_num / M_exact - 1))
err_Phi = np.max(np.abs(Phi_num / Phi_exact - 1))
print(f'max relative error: M(r) {err_M:.1e},  Phi(r) {err_Phi:.1e}')
assert err_M < 1e-6 and err_Phi < 1e-6

Note the shapes: the density is flat inside $b$ (a *core*) and falls as $r^{-5}$ outside;
the enclosed mass grows as $r^3$ in the core and saturates at $M$; the potential is finite
at the centre, $\Phi(0) = -GM/b$.

### Circular speed

A star on a circular orbit needs centripetal acceleration $v^2/r$ equal to the inward pull:

$$ v_c^2(r) = r\,\frac{d\Phi}{dr} = \frac{G M(r)}{r}. $$

Compare three models of total mass 1:
- **point mass** (Kepler), $\Phi = -GM/r$: $v_c \propto r^{-1/2}$ at all radii;
- **homogeneous sphere** of radius $a$: $M(r) = M r^3/a^3$ inside, so $v_c \propto r$ — *solid-body* rotation;
- **Plummer**: solid-body in the core, Keplerian far outside, with a peak in between.

In [ ]:
def vc_kepler(r):      return np.sqrt(G * M / r)
def vc_homog(r, a=1):  return np.sqrt(G * M * np.minimum(r, a)**3 / a**3 / r)
def vc_plummer(r):     return np.sqrt(G * M * r**2 / (r**2 + b**2)**1.5)

r = np.linspace(0.01, 6, 400)
plt.figure(figsize=(6, 4))
plt.plot(r, vc_kepler(r), label='point mass')
plt.plot(r, vc_homog(r), label='homogeneous sphere, $a=1$')
plt.plot(r, vc_plummer(r), label='Plummer, $b=1$')
plt.ylim(0, 1.5); plt.xlabel('$r$'); plt.ylabel('$v_c$'); plt.legend(); plt.show()

# where does the Plummer curve peak?  Analytically: d(v_c^2)/dr = 0  =>  r = sqrt(2) b
r_peak = minimize_scalar(lambda x: -vc_plummer(x), bounds=(0.1, 5), method='bounded',
                         options={'xatol': 1e-10}).x
print(f'numerical peak at r = {r_peak:.8f},  sqrt(2) b = {np.sqrt(2) * b:.8f}')
assert abs(r_peak - np.sqrt(2) * b) < 1e-6

Real galaxies show *flat* rotation curves at large radius — neither of these shapes.
That observation is one of the main pieces of evidence for dark matter halos, which we
meet in Chapter 4.

## 1.3 Energy, angular momentum and the effective potential

In a spherical potential a star conserves its **energy** and its **angular-momentum vector**
(the force is radial, so it exerts no torque):

$$ E = \tfrac12 v^2 + \Phi(r), \qquad \mathbf L = \mathbf r \times \mathbf v. $$

Since $\mathbf L$ is fixed, the orbit stays in a plane. Split the velocity into a radial part
$v_r$ and a tangential part $v_t = L/r$:

$$ E = \tfrac12 v_r^2 + \underbrace{\Phi(r) + \frac{L^2}{2r^2}}_{\Phi_{\rm eff}(r)} . $$

This is a one-dimensional problem: a bead of energy $E$ sliding in the **effective potential**
$\Phi_{\rm eff}$. Because $v_r^2 \ge 0$, the star is confined to where $\Phi_{\rm eff}(r) \le E$.
The two radii where $\Phi_{\rm eff}(r) = E$ are the **pericentre** $r_p$ and **apocentre** $r_a$.
The minimum of $\Phi_{\rm eff}$ is the circular orbit with that $L$.

In [ ]:
def phi_kepler(r):  return -G * M / r
def phi_plummer(r): return -G * M / np.sqrt(r**2 + b**2)

def phi_eff(r, L, phi):
    return phi(r) + L**2 / (2 * r**2)

r = np.linspace(0.05, 5, 500)
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for L in [0.4, 0.6, 0.8, 1.0]:
    ax[0].plot(r, phi_eff(r, L, phi_kepler), label=f'$L={L}$')
    ax[1].plot(r, phi_eff(r, L, phi_plummer), label=f'$L={L}$')
ax[0].plot(r, phi_kepler(r), 'k--', label=r'$\Phi$ ($L=0$)')
ax[1].plot(r, phi_plummer(r), 'k--', label=r'$\Phi$ ($L=0$)')
ax[0].set_ylim(-3, 1); ax[1].set_ylim(-1.1, 0.3)
ax[0].set_title('point mass'); ax[1].set_title('Plummer sphere')
for a in ax: a.set_xlabel('$r$'); a.set_ylabel(r'$\Phi_{\rm eff}$'); a.legend()
plt.show()

The $L^2/2r^2$ term is a **centrifugal barrier**: the more angular momentum, the further the
star is kept from the centre.

### Turning points of a Kepler orbit

For a point mass the answer is known: an orbit with semi-major axis $a$ and eccentricity $e$
has $E = -GM/2a$, $L^2 = GMa(1-e^2)$, and turns at $r_{p,a} = a(1 \mp e)$. We find the roots
of $\Phi_{\rm eff}(r) - E$ with `brentq`, which needs an interval where the function changes
sign: below the circular radius $r_c = L^2/GM$ for the pericentre and above it for the apocentre.

In [ ]:
def turning_points(E, L, phi, r_circ, r_max=1e3):
    """Pericentre and apocentre: roots of Phi_eff(r) = E either side of the circular radius."""
    f = lambda r: phi_eff(r, L, phi) - E
    return brentq(f, 1e-8, r_circ, xtol=1e-14), brentq(f, r_circ, r_max, xtol=1e-14)

a, e = 1.0, 0.7
E = -G * M / (2 * a)
L = np.sqrt(G * M * a * (1 - e**2))
rp, ra = turning_points(E, L, phi_kepler, r_circ=L**2 / (G * M))
print(f'pericentre {rp:.12f}   expected a(1-e) = {a * (1 - e):.12f}')
print(f'apocentre  {ra:.12f}   expected a(1+e) = {a * (1 + e):.12f}')
assert abs(rp - a * (1 - e)) < 1e-8 * a and abs(ra - a * (1 + e)) < 1e-8 * a

r = np.linspace(0.1, 2.5, 400)
plt.figure(figsize=(6, 4))
plt.plot(r, phi_eff(r, L, phi_kepler), 'k-', label=r'$\Phi_{\rm eff}$')
plt.axhline(E, color='C1', label='$E$')
plt.plot([rp, ra], [E, E], 'o', color='C3', label='turning points')
plt.ylim(-2, 0.5); plt.xlabel('$r$'); plt.ylabel('energy'); plt.legend()
plt.title(f'Kepler orbit, $a={a}$, $e={e}$'); plt.show()

### The same in the Plummer potential

For the Plummer sphere there is no simple formula, but the method is identical. We pick a
star at $r=1$ moving tangentially with 70% of the circular speed, so it starts at apocentre.

In [ ]:
r0, v0 = 1.0, 0.7 * vc_plummer(1.0)
L = r0 * v0
E = 0.5 * v0**2 + phi_plummer(r0)
r_circ = minimize_scalar(lambda r: phi_eff(r, L, phi_plummer), bounds=(1e-3, 10), method='bounded').x
rp, ra = turning_points(E, L, phi_plummer, r_circ)
print(f'circular radius for this L: {r_circ:.4f}')
print(f'pericentre {rp:.6f}, apocentre {ra:.6f}')
assert abs(ra - r0) < 1e-8          # we launched it tangentially, so r0 is a turning point

In Chapter 2 we will integrate this orbit numerically and check that it really oscillates
between these two radii.

## 1.4 The virial theorem

For a self-gravitating system in a steady state, the total kinetic energy $K$ and potential
energy $W$ obey

$$ 2K + W = 0 . $$

The potential energy of a spherical system is the work needed to assemble it shell by shell:

$$ W = -4\pi G \int_0^\infty \rho(r)\, M(r)\, r\, dr . $$

For the Plummer sphere $W = -\dfrac{3\pi}{32}\dfrac{GM^2}{b}$, so the virial theorem predicts
$K = \dfrac{3\pi}{64}\dfrac{GM^2}{b}$ and a mean-square speed $\langle v^2\rangle = 2K/M$.
This lets us estimate a system's mass from its size and velocity spread — the idea behind
the first evidence for dark matter in galaxy clusters (Zwicky 1933).

In [ ]:
W_num = -4 * np.pi * G * quad(lambda x: rho_plummer(x) * M * x**3 / (x**2 + b**2)**1.5 * x, 0, np.inf)[0]
W_exact = -3 * np.pi / 32 * G * M**2 / b
print(f'W numerical {W_num:.10f},  formula {W_exact:.10f}')
assert abs(W_num / W_exact - 1) < 1e-6
print(f'virial prediction: K = {-W_exact / 2:.4f},  rms speed = {np.sqrt(-W_exact / M):.4f}')

We check the kinetic-energy side in Chapter 8, by sampling stellar velocities from the
Plummer model's distribution function.

## Summary

- Only the mass inside $r$ pulls a star at $r$ inwards: $g = -GM(r)/r^2$.
- Density → enclosed mass → potential → circular speed, all by one-dimensional integrals.
- In a spherical potential, $E$ and $L$ reduce the orbit to motion in $\Phi_{\rm eff}(r)$;
  its turning points are the pericentre and apocentre.
- The virial theorem, $2K + W = 0$, links a system's mass, size and velocities.

## Exercises

1. Repeat §1.1 with a shell of 200 points. How large is the field inside now? Plot the error
   in $g$ against $N$ on log axes and find the scaling.
2. The **Hernquist** model has $\rho = \dfrac{M b}{2\pi r (r+b)^3}$. Compute $M(r)$ and $\Phi(r)$
   numerically with the functions above and compare with $M(r) = M r^2/(r+b)^2$,
   $\Phi = -GM/(r+b)$. Where does its circular speed peak?
3. For a homogeneous sphere of radius $a$, show that $\Phi = -\dfrac{GM}{2a^3}(3a^2 - r^2)$ inside.
   Plot $\Phi_{\rm eff}$ for $L = 0.3$ and find the turning points of an orbit with $E = -1.2$.
4. For the Plummer orbit in §1.3, find $L$ such that $r_a/r_p = 10$ for a star launched
   tangentially at $r = 1$.
5. Verify $W = -\tfrac{3\pi}{32} GM^2/b$ by integrating $\tfrac12\int \rho\,\Phi\, dV$ instead.
   Why does it give the same answer?

## References

- J. Binney & S. Tremaine, *Galactic Dynamics*, 2nd ed. (Princeton, 2008), §2.2 and §3.1.
- J. Bovy, *Dynamics and Astrophysics of Galaxies*, chapters 2–4: https://galaxiesbook.org
- H. C. Plummer (1911), MNRAS 71, 460: https://doi.org/10.1093/mnras/71.5.460
- F. Zwicky (1933), Helvetica Physica Acta 6, 110: https://ui.adsabs.harvard.edu/abs/1933AcHPh...6..110Z